In [0]:
###
STORAGE_ACCOUNT   = "dltlearn"         
RAW_CONTAINER     = "raw"                                             

# Base paths
RAW_BASE_PATH     = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/raw"
BRONZE_BASE_PATH  = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"

catalog_name = 'test'
schema = 'bronze'

# BRONZE_DATABASE   = f:"{catalog_name}+'.'+{schema}" 
BRONZE_DATABASE   = catalog_name+'.'+schema

from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

def ingest_bronze_table(file_name: str,table_name: str):
   file_path = f"{RAW_BASE_PATH}/{file_name}"
   delta_path = f"{BRONZE_BASE_PATH}/{table_name}"
   print(f"Loading filename is {file_name}")
   try:
       df = spark.read.format('CSV').option('header',True).option('InferSchema',False).option('escape','"')\
       .option('multiline',True).load(file_path)
       rows = df.count()
       columns = df.columns
       print(f"rows {rows} and columns {columns}")

       df = df.withColumn("ingestion_timestamp",current_timestamp())\
              .withColumn("source_file",lit(file_name))
               
       df.write.format("delta").mode("overwrite").save(delta_path)  


       spark.sql(f"create database if not exists {BRONZE_DATABASE}")
       spark.sql(f"""create table if not exists {BRONZE_DATABASE}.{table_name}
                     using delta
                     location '{delta_path}'""")
       return df
   except Exception as e:
       print(f"failed file {file_name}")
       print(f"error is {str(e)}")
       return None 

**Now creating tables**

In [0]:
df_customer = ingest_bronze_table(file_name ='olist_customers_dataset.csv',table_name='customer')
df_geolocation = ingest_bronze_table(file_name ='olist_geolocation_dataset.csv',table_name='geolocation')
df_orderitems = ingest_bronze_table(file_name ='olist_order_items_dataset.csv',table_name='orderitems')
df_orderpayments = ingest_bronze_table(file_name ='olist_order_payments_dataset.csv',table_name='orderpayments')
df_orderreviews = ingest_bronze_table(file_name ='olist_order_reviews_dataset.csv',table_name='orderreviews')
df_orders = ingest_bronze_table(file_name ='olist_orders_dataset.csv',table_name='orders')
df_products = ingest_bronze_table(file_name ='olist_orders_dataset.csv',table_name='products')
df_sellers = ingest_bronze_table(file_name ='olist_sellers_dataset.csv',table_name='sellers')
df_productcategory = ingest_bronze_table(file_name ='product_category_name_translation.csv',table_name='product_category')
